# Filas: conceito e implementação — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

A aula passada terminou com uma pergunta: a pilha devolve tudo na ordem inversa; e quando é preciso
atender na ordem de **chegada**? Este tutorial retoma o programa `batata.c` visto hoje — a roda da batata
quente, em que a batata passa $k$ vezes e quem fica com ela sai — e segue o caminho dos slides: o que é
uma fila, a fila num vetor que "anda" até bater na parede, a fila circular, o vetor circular que cresce
sem embaralhar, a fila numa lista e o custo de cada versão. Depois de cada bloco há um exercício com
teste automático; no fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Descrever uma fila pelas suas operações e reconhecer problemas que pedem uma
- Explicar por que uma fila num vetor com `ini` e `fim` que só crescem "enche" com espaço sobrando
- Implementar a fila circular com `% cap` e um contador `n`, distinguindo fila cheia de fila vazia
- Fazer o vetor circular crescer copiando **na ordem da fila**
- Implementar a fila numa lista simples com `ini` e `fim`, inclusive o caso da fila que esvazia
- Comparar o custo de deslocar, dar a volta e encadear

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: algum laço está fazendo trabalho à toa?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip().split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
        if r.stderr:
            print(r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Antes de executar, escreva a saída que
você espera para as três rodas: com $k = 0$ a batata nem passa; com $n = 5$ e $k = 2$, quem sai primeiro?

In [ ]:
%%writefile batata.c
/* batata.c --- quem sobra na roda da batata quente?
   n criancas em roda; a batata passa k vezes e quem fica com ela sai.
   uso: ./batata          (tres rodas de exemplo)
        ./batata n k      (uma roda) */
#include <stdio.h>
#include <stdlib.h>

#define MAX 100

typedef struct {
    int v[MAX];    /* a frente e v[ini]; o proximo entra em v[fim] */
    int ini, fim;
} Fila;

void enfileira(Fila *f, int x)
{
    if (f->fim == MAX) {
        fprintf(stderr, "\nenfileira: fila cheia\n");
        exit(1);
    }
    f->v[f->fim] = x;
    f->fim++;
}

int desenfileira(Fila *f)
{
    if (f->ini == f->fim) {
        fprintf(stderr, "\ndesenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini++;
    return x;
}

int tamanho(const Fila *f)
{
    return f->fim - f->ini;
}

/* imprime a ordem de saida e devolve quem sobrou */
int roda(int n, int k)
{
    Fila f = {.ini = 0, .fim = 0};
    for (int i = 1; i <= n; i++)
        enfileira(&f, i);
    while (tamanho(&f) > 1) {
        for (int j = 0; j < k; j++)
            enfileira(&f, desenfileira(&f));   /* passa a batata */
        printf(" %d", desenfileira(&f));        /* quem ficou com ela sai */
        fflush(stdout);
    }
    return desenfileira(&f);
}

int main(int argc, char *argv[])
{
    if (argc == 3) {
        int n = atoi(argv[1]), k = atoi(argv[2]);
        printf("n = %d, k = %d: sai", n, k);
        int fica = roda(n, k);
        printf(" | fica %d\n", fica);
        return 0;
    }
    int casos[][2] = {{6, 0}, {5, 2}, {7, 1}};
    for (int i = 0; i < 3; i++) {
        printf("n = %d, k = %d: sai", casos[i][0], casos[i][1]);
        int fica = roda(casos[i][0], casos[i][1]);
        printf(" | fica %d\n", fica);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra batata.c -o batata && ./batata

## 1. O que é uma fila

Uma **fila** é uma sequência em que as inserções acontecem numa ponta, o *fim*, e as remoções na outra,
a *frente*: o primeiro a entrar é o primeiro a sair (*FIFO*, *first in, first out*).

| operação | o que faz |
|---|---|
| `enfileira(f, x)` | põe `x` no fim |
| `desenfileira(f)` | tira e devolve a frente |
| `frente(f)` | devolve a frente sem tirar |
| `vazia(f)` | a fila não tem nada? |

Em `batata.c`, a fila guarda a roda inteira: quem está com a batata na frente, quem acabou de passá-la
no fim. Passar a batata é `enfileira(&f, desenfileira(&f))` — a criança da frente vai para o fim, e a
roda gira sem que ninguém mude de lugar em relação aos outros. Esta roda é o **problema de Josephus**.

O mesmo giro aparece no sistema operacional: o escalonador *round-robin* dá a cada programa uma fatia de
tempo e, se ele não terminou, o põe de volta no fim da fila de prontos.

### Exercício 1 — escalonador *round-robin*

O processo `i` (de 0 a `n-1`) precisa de `t[i]` unidades de tempo de processador. O da frente da fila
recebe até `q` unidades; se ainda não terminou, volta para o fim. O relógio começa em 0. Imprima
`P<i> termina em <instante>` na ordem em que os processos terminam. Use a fila de `batata.c`, que já
está no arquivo.

In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

#define MAX 1000

typedef struct {
    int v[MAX];    /* a frente e v[ini]; o proximo entra em v[fim] */
    int ini, fim;
} Fila;

void enfileira(Fila *f, int x)
{
    if (f->fim == MAX) {
        fprintf(stderr, "enfileira: fila cheia\n");
        exit(1);
    }
    f->v[f->fim] = x;
    f->fim++;
}

int desenfileira(Fila *f)
{
    if (f->ini == f->fim) {
        fprintf(stderr, "desenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini++;
    return x;
}

int vazia(const Fila *f)
{
    return f->ini == f->fim;
}

/* Exercicio 1: o processo i precisa de t[i] unidades de tempo; o da frente
   recebe ate q unidades e, se nao terminou, volta para o fim. Imprima
   "P<i> termina em <instante>" na ordem de termino; o relogio comeca em 0.
   TODO: implemente escalona. */
void escalona(int n, const int t[], int q)
{
    /* TODO: implemente aqui */
    (void) n;
    (void) t;
    (void) q;
}

int main(void)
{
    int t1[] = {5, 2, 4}, t2[] = {3, 3, 3}, t3[] = {1, 7, 2, 6};
    printf("q = 2, t = {5, 2, 4}\n");
    escalona(3, t1, 2);
    printf("q = 3, t = {3, 3, 3}\n");
    escalona(3, t2, 3);
    printf("q = 3, t = {1, 7, 2, 6}\n");
    escalona(4, t3, 3);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex1.c -o ex1
testa('./ex1', '''q = 2, t = {5, 2, 4}
P1 termina em 4
P2 termina em 10
P0 termina em 11
q = 3, t = {3, 3, 3}
P0 termina em 3
P1 termina em 6
P2 termina em 9
q = 3, t = {1, 7, 2, 6}
P0 termina em 1
P2 termina em 6
P3 termina em 15
P1 termina em 16''')

## 2. Fila num vetor

No vetor, a frente é `v[ini]` e o próximo entra em `v[fim]`. Enfileirar escreve em `v[fim]` e
incrementa `fim`; desenfileirar lê `v[ini]` e incrementa `ini`. Nenhuma operação tem laço, e
`batata.c` já confere fila cheia e fila vazia — a lição da aula de pilhas.

Mas, ao contrário do topo da pilha, que anda para os dois lados, `ini` e `fim` **só aumentam**. A roda de
41 crianças nunca tem mais de 41 na fila, e `MAX` é 100. Antes de executar: cabe? Quantas vezes
`enfileira` é chamada até sobrar uma criança?

In [ ]:
!./batata 41 2

Saíram 29 crianças e a fila "encheu" com 12 na roda. Cada criança que entra e cada passada da batata é
um `enfileira`: são $n + k(n-1) = 41 + 2 \cdot 40 = 121$ ao todo, e `fim` só cresce. A fila foi andando
para a direita e bateu na parede, deixando para trás posições que ninguém mais usa.

A célula abaixo confere a conta: com `MAX = 120` ainda falta espaço; com `MAX = 121`, a roda termina.

In [ ]:
for m in [120, 121]:
    print(f'MAX = {m}:')
    !sed "s/#define MAX 100/#define MAX {m}/" batata.c > batata_max.c
    !gcc -Wall -Wextra batata_max.c -o batata_max && ./batata_max 41 2 2>&1 | tail -c 40

Uma primeira saída é a da fila de banco: quando o primeiro é atendido, todos dão um passo para a frente.
Sem `ini`, a frente é sempre `v[0]` — e cada `desenfileira` copia a fila inteira. Veremos o custo disso
na seção 6.

## 3. A fila circular

A saída boa é dobrar o vetor em anel: depois de `v[MAX-1]` vem `v[0]`. Quando o fim chega à última
posição, continua no começo, no espaço que a frente já liberou. "Dar a volta" é o resto da divisão:
`(ini + 1) % MAX`.

Só que, com `ini` e `fim` apenas, a fila **cheia** e a **vazia** viram o mesmo estado: `ini == fim`. A
solução usada aqui é guardar `n`, o número de elementos — vazia é `n == 0`, cheia é `n == MAX`, e o
próximo entra em `v[(ini + n) % MAX]`. (O livro de Cormen prefere deixar sempre uma posição vazia.)

Em `batata_circular.c`, só a fila mudou por dentro; `roda` e `main` são os mesmos. Agora `MAX` é 41.

In [ ]:
%%writefile batata_circular.c
/* batata_circular.c --- a roda de batata.c com a fila circular:
   o fim volta ao inicio do vetor. MAX agora e so 41. */
#include <stdio.h>
#include <stdlib.h>

#define MAX 41

typedef struct {
    int v[MAX];    /* a frente e v[ini]; os n seguintes, dando a volta */
    int ini, n;
} Fila;

void enfileira(Fila *f, int x)
{
    if (f->n == MAX) {
        fprintf(stderr, "\nenfileira: fila cheia\n");
        exit(1);
    }
    f->v[(f->ini + f->n) % MAX] = x;    /* o fim da a volta */
    f->n++;
}

int desenfileira(Fila *f)
{
    if (f->n == 0) {
        fprintf(stderr, "\ndesenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % MAX;        /* o inicio tambem */
    f->n--;
    return x;
}

int tamanho(const Fila *f)
{
    return f->n;
}

int roda(int n, int k)
{
    Fila f = {.ini = 0, .n = 0};
    for (int i = 1; i <= n; i++)
        enfileira(&f, i);
    while (tamanho(&f) > 1) {
        for (int j = 0; j < k; j++)
            enfileira(&f, desenfileira(&f));
        printf(" %d", desenfileira(&f));
    }
    return desenfileira(&f);
}

int main(int argc, char *argv[])
{
    int n = (argc > 2) ? atoi(argv[1]) : 41;
    int k = (argc > 2) ? atoi(argv[2]) : 2;
    printf("n = %d, k = %d: sai", n, k);
    int fica = roda(n, k);
    printf(" | fica %d\n", fica);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra batata_circular.c -o batata_circular && ./batata_circular 41 2

Os 121 `enfileira` cabem num vetor de 41. Conta a lenda que Josephus e um amigo, numa roda de 41 em que
cada terceiro saía, escolheram os lugares 16 e 31 — os dois últimos a sair.

### Exercício 2 — uma fila circular que avisa

Implemente a fila circular com `ini` e `n`:

- `int enfileira(Fila *f, int x)`: devolve 1 se enfileirou e 0 se a fila estava cheia;
- `int desenfileira(Fila *f, int *x)`: devolve 1 e a frente em `*x`, ou 0 se a fila estava vazia.

Em caso de falha, a fila não pode mudar. `MAX` é 4 de propósito: o teste faz o fim dar a volta. Ele roda
com `-fsanitize=address`.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

#define MAX 4   /* pequeno de proposito */

typedef struct {
    int v[MAX];    /* a frente e v[ini]; os n seguintes, dando a volta */
    int ini, n;
} Fila;

/* Exercicio 2: enfileira devolve 1 se conseguiu e 0 se a fila esta cheia;
   desenfileira devolve 1 e a frente em *x, ou 0 se a fila esta vazia.
   Em caso de falha, a fila nao pode mudar.
   TODO: implemente enfileira e desenfileira. */
int enfileira(Fila *f, int x)
{
    /* TODO: implemente aqui */
    (void) f;
    (void) x;
    return 0;
}

int desenfileira(Fila *f, int *x)
{
    /* TODO: implemente aqui */
    (void) f;
    (void) x;
    return 0;
}

int main(void)
{
    Fila f = {.ini = 0, .n = 0};
    int x;
    printf("desenfileira na vazia: %d\n", desenfileira(&f, &x));
    printf("enfileira 1 a 5:");
    for (int i = 1; i <= 5; i++)
        printf(" %d", enfileira(&f, i));
    printf("\nsai:");
    for (int i = 0; i < 2; i++)
        if (desenfileira(&f, &x))
            printf(" %d", x);
    printf("\nenfileira 6, 7, 8: %d %d %d\n", enfileira(&f, 6), enfileira(&f, 7),
           enfileira(&f, 8));
    printf("ini = %d, n = %d, v = {%d, %d, %d, %d}\n", f.ini, f.n, f.v[0], f.v[1],
           f.v[2], f.v[3]);
    printf("sai:");
    while (desenfileira(&f, &x))
        printf(" %d", x);
    printf("\nini = %d, n = %d\n", f.ini, f.n);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex2.c -o ex2
testa('./ex2', '''desenfileira na vazia: 0
enfileira 1 a 5: 1 1 1 1 0
sai: 1 2
enfileira 6, 7, 8: 1 1 0
ini = 2, n = 4, v = {6, 7, 3, 4}
sai: 3 4 6 7
ini = 2, n = 0''')

## 4. Quando o vetor circular enche

Na pilha, crescer era um `realloc` e pronto. `dobra_errada.c` faz o mesmo com a fila circular: começa
com `cap = 4`, enfileira de 1 a 4, desenfileira dois, enfileira 5 e 6 — que dão a volta — e enfileira o 7,
que faz o vetor dobrar. Depois desenfileira tudo. Desenhe o vetor antes e depois do `realloc` e preveja a
saída.

In [ ]:
%%writefile dobra_errada.c
/* dobra_errada.c --- fila circular que dobra com um realloc e nada mais */
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;
    int ini, n, cap;
} Fila;

void enfileira(Fila *f, int x)
{
    if (f->n == f->cap) {                 /* cheia: dobra */
        f->cap *= 2;
        f->v = realloc(f->v, f->cap * sizeof *f->v);
        if (f->v == NULL) { perror("realloc"); exit(1); }
    }
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

int desenfileira(Fila *f)
{
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

int main(void)
{
    Fila f = {.ini = 0, .n = 0, .cap = 4};
    f.v = malloc(f.cap * sizeof *f.v);
    if (f.v == NULL) { perror("malloc"); exit(1); }

    for (int x = 1; x <= 4; x++) enfileira(&f, x);   /* 1 2 3 4 */
    desenfileira(&f);                                /* sai 1   */
    desenfileira(&f);                                /* sai 2   */
    enfileira(&f, 5);                                /* 3 4 5   */
    enfileira(&f, 6);                                /* 3 4 5 6: cheia */
    enfileira(&f, 7);                                /* dobra   */

    printf("cap = %d, ini = %d, n = %d\nsai:", f.cap, f.ini, f.n);
    while (f.n > 0)
        printf(" %d", desenfileira(&f));
    printf("\n");
    free(f.v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra dobra_errada.c -o dobra_errada && ./dobra_errada
!gcc -g -fsanitize=address dobra_errada.c -o dobra_errada-asan && ./dobra_errada-asan

O 5 e o 6 sumiram. Antes do 7, o vetor era `{5, 6, 3, 4}` com `ini = 2`: na fila, o 5 vinha **depois** do
4, dando a volta. O `realloc` copia as posições 0 a 3 para as posições 0 a 3 do bloco novo — mas, com
`cap = 8`, a posição seguinte a `v[3]` passou a ser `v[4]`, que é memória nova, nunca escrita. A conta
`(ini + i) % cap` mudou de significado quando `cap` mudou.

O *sanitizer* não acusa nada: todas as posições lidas estão dentro do bloco. Ele só preenche a memória
nova com o padrão `0xBEBEBEBE` (−1094795586), e o padrão aparece na saída.

Para crescer é preciso **desenrolar o anel**: copiar na ordem da fila, da frente para o fim, para o
começo do vetor novo, e fazer `ini = 0`. Ainda são $n$ cópias por crescimento; dobrando, o custo por
`enfileira` continua constante, amortizado, exatamente como na pilha.

### Exercício 3 — crescer sem embaralhar

Implemente `cresce`: aloque um vetor com o dobro da capacidade, copie os `n` elementos **na ordem da
fila** para as posições `0..n-1`, libere o antigo e acerte `ini` e `cap`. O teste repete o cenário de
`dobra_errada.c` e depois intercala 1000 entradas com saídas, conferindo a ordem.

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;
    int ini, n, cap;
} Fila;

/* Exercicio 3: troque f->v por um vetor com o dobro da capacidade, com os n
   elementos na ordem da fila nas posicoes 0..n-1 (confira o NULL do malloc
   e libere o vetor antigo); depois acerte ini e cap.
   TODO: implemente cresce. */
void cresce(Fila *f)
{
    /* TODO: implemente aqui */
    (void) f;
}

void enfileira(Fila *f, int x)
{
    if (f->n == f->cap)
        cresce(f);
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

int desenfileira(Fila *f)
{
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

void inicia(Fila *f)
{
    f->ini = f->n = 0;
    f->cap = 4;
    f->v = malloc(f->cap * sizeof *f->v);
    if (f->v == NULL) { perror("malloc"); exit(1); }
}

int main(void)
{
    Fila f;
    inicia(&f);
    for (int x = 1; x <= 4; x++) enfileira(&f, x);
    desenfileira(&f);
    desenfileira(&f);
    enfileira(&f, 5);
    enfileira(&f, 6);
    enfileira(&f, 7);
    printf("cap = %d, ini = %d, n = %d\nsai:", f.cap, f.ini, f.n);
    while (f.n > 0)
        printf(" %d", desenfileira(&f));
    printf("\n");
    free(f.v);

    inicia(&f);
    int proximo = 1, ok = 1, saidas = 0;
    for (int x = 1; x <= 1000; x++) {
        enfileira(&f, x);
        if (x % 3 == 0) {               /* de vez em quando, alguem sai */
            ok = ok && desenfileira(&f) == proximo++;
            saidas++;
        }
    }
    printf("1000 entradas, %d saidas no meio: cap = %d, n = %d\n", saidas, f.cap, f.n);
    while (f.n > 0)
        ok = ok && desenfileira(&f) == proximo++;
    printf("ordem %s\n", ok && proximo == 1001 ? "ok" : "ERRADA");
    free(f.v);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex3.c -o ex3
testa('./ex3', '''cap = 8, ini = 0, n = 5
sai: 3 4 5 6 7
1000 entradas, 333 saidas no meio: cap = 1024, n = 667
ordem ok''')

## 5. Fila numa lista

Na lista simples com `ini` e `fim`, três das quatro operações nas pontas são baratas: inserir no início,
remover o primeiro e inserir depois do último. Remover o **último** exige achar o penúltimo — percorrer a
lista toda. Por isso a fila sai do primeiro nó (a frente) e entra depois do último (o fim). Diferente da
pilha, que só usava uma ponta, a fila precisa de `fim` — sem ele, cada `enfileira` percorreria a lista.

O caso delicado é a fila que esvazia: quando sai o último nó, `ini` vira `NULL` e `fim` precisa virar
`NULL` também. `esvazia.c` esquece disso. Preveja: depois de enfileirar 1 e 2, desenfileirar os dois e
enfileirar o 3, a fila está vazia?

In [ ]:
%%writefile esvazia.c
/* esvazia.c --- a fila em lista sem atualizar fim quando ela esvazia */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

typedef struct {
    No *ini, *fim;
} Fila;

void enfileira(Fila *f, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) exit(1);
    novo->valor = x;
    novo->prox = NULL;
    if (f->fim == NULL)
        f->ini = novo;
    else
        f->fim->prox = novo;
    f->fim = novo;
}

int desenfileira(Fila *f)
{
    No *velho = f->ini;
    int x = velho->valor;
    f->ini = velho->prox;
    /* sumiu: if (f->ini == NULL) f->fim = NULL; */
    free(velho);
    return x;
}

int vazia(const Fila *f)
{
    return f->ini == NULL;
}

int main(void)
{
    Fila f = {NULL, NULL};
    enfileira(&f, 1);
    enfileira(&f, 2);
    printf("sai %d\n", desenfileira(&f));
    printf("sai %d\n", desenfileira(&f));
    enfileira(&f, 3);
    printf("vazia? %s\n", vazia(&f) ? "sim" : "nao");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra esvazia.c -o esvazia && ./esvazia
!gcc -g -fsanitize=address esvazia.c -o esvazia-asan && ./esvazia-asan 2>&1 | head -16

Ao sair o 2, `ini` virou `NULL`, mas `fim` continuou apontando para o nó do 2 — já devolvido pelo `free`.
Ao enfileirar o 3, `fim != NULL`: o programa escreve `fim->prox` em memória liberada (o
**heap-use-after-free** do ASan) e deixa `ini` em `NULL`. O 3 foi alocado, mas ninguém chega até ele.

Como na aula de pilhas, as operações vão para um cabeçalho, `fila.h`, que só **declara** `struct fila`.
Há duas implementações: `fila_vetor.c`, a circular que cresce desenrolando o anel, e `fila_lista.c`.

In [ ]:
%%writefile fila.h
/* fila.h --- o que o usuario da fila precisa saber, e nada mais */
#ifndef FILA_H
#define FILA_H

typedef struct fila Fila;     /* os campos ficam no .c */

Fila *fila_cria(void);
void fila_libera(Fila *f);

void enfileira(Fila *f, int x);   /* entra no fim */
int desenfileira(Fila *f);        /* sai da frente */
int frente(const Fila *f);        /* devolve a frente sem tirar */
int vazia(const Fila *f);
int tamanho(const Fila *f);

#endif

In [ ]:
%%writefile fila_vetor.c
/* fila_vetor.c --- fila circular num vetor que dobra quando enche */
#include <stdio.h>
#include <stdlib.h>
#include "fila.h"

struct fila {
    int *v;    /* a frente e v[ini]; os n seguintes, dando a volta */
    int ini;   /* posicao da frente */
    int n;     /* quantos elementos ha */
    int cap;   /* quantos cabem em v */
};

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

Fila *fila_cria(void)
{
    Fila *f = aloca(NULL, sizeof *f);
    f->ini = 0;
    f->n = 0;
    f->cap = 4;
    f->v = aloca(NULL, f->cap * sizeof *f->v);
    return f;
}

void fila_libera(Fila *f)
{
    free(f->v);
    free(f);
}

/* troca v por um vetor com o dobro do tamanho, ja na ordem: frente em 0 */
static void cresce(Fila *f)
{
    int *novo = aloca(NULL, 2 * f->cap * sizeof *novo);
    for (int i = 0; i < f->n; i++)
        novo[i] = f->v[(f->ini + i) % f->cap];
    free(f->v);
    f->v = novo;
    f->ini = 0;
    f->cap *= 2;
}

void enfileira(Fila *f, int x)
{
    if (f->n == f->cap)
        cresce(f);
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

int desenfileira(Fila *f)
{
    if (f->n == 0) {
        fprintf(stderr, "desenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

int frente(const Fila *f)
{
    if (f->n == 0) {
        fprintf(stderr, "frente: fila vazia\n");
        exit(1);
    }
    return f->v[f->ini];
}

int vazia(const Fila *f)
{
    return f->n == 0;
}

int tamanho(const Fila *f)
{
    return f->n;
}

In [ ]:
%%writefile fila_lista.c
/* fila_lista.c --- fila numa lista encadeada: sai do primeiro no,
   entra depois do ultimo */
#include <stdio.h>
#include <stdlib.h>
#include "fila.h"

typedef struct no {
    int valor;
    struct no *prox;   /* o de tras, ou NULL */
} No;

struct fila {
    No *ini;           /* a frente: primeiro no, ou NULL */
    No *fim;           /* o ultimo no, ou NULL */
    int n;
};

Fila *fila_cria(void)
{
    Fila *f = malloc(sizeof *f);
    if (f == NULL) {
        perror("malloc");
        exit(1);
    }
    f->ini = f->fim = NULL;
    f->n = 0;
    return f;
}

void fila_libera(Fila *f)
{
    while (!vazia(f))
        desenfileira(f);
    free(f);
}

void enfileira(Fila *f, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = NULL;
    if (f->fim == NULL)          /* fila vazia: o novo e tambem a frente */
        f->ini = novo;
    else
        f->fim->prox = novo;
    f->fim = novo;
    f->n++;
}

int desenfileira(Fila *f)
{
    if (f->ini == NULL) {
        fprintf(stderr, "desenfileira: fila vazia\n");
        exit(1);
    }
    No *velho = f->ini;
    int x = velho->valor;
    f->ini = velho->prox;
    if (f->ini == NULL)          /* saiu o ultimo: nao ha mais fim */
        f->fim = NULL;
    free(velho);
    f->n--;
    return x;
}

int frente(const Fila *f)
{
    if (f->ini == NULL) {
        fprintf(stderr, "frente: fila vazia\n");
        exit(1);
    }
    return f->ini->valor;
}

int vazia(const Fila *f)
{
    return f->ini == NULL;
}

int tamanho(const Fila *f)
{
    return f->n;
}

In [ ]:
%%writefile batata_tad.c
/* batata_tad.c --- a roda de batata.c, usando so fila.h
   gcc batata_tad.c fila_vetor.c   ou   gcc batata_tad.c fila_lista.c */
#include <stdio.h>
#include <stdlib.h>
#include "fila.h"

/* devolve quem sobra; se imprime, mostra a ordem de saida */
int roda(int n, int k, int imprime)
{
    Fila *f = fila_cria();
    for (int i = 1; i <= n; i++)
        enfileira(f, i);
    while (tamanho(f) > 1) {
        for (int j = 0; j < k; j++)
            enfileira(f, desenfileira(f));
        int sai = desenfileira(f);
        if (imprime)
            printf(" %d", sai);
    }
    int fica = desenfileira(f);
    fila_libera(f);
    return fica;
}

int main(void)
{
    int casos[][2] = {{6, 0}, {5, 2}, {7, 1}};
    for (int i = 0; i < 3; i++) {
        printf("n = %d, k = %d: sai", casos[i][0], casos[i][1]);
        int fica = roda(casos[i][0], casos[i][1], 1);
        printf(" | fica %d\n", fica);
    }
    printf("n = 41, k = 2: fica %d\n", roda(41, 2, 0));
    printf("n = 100000, k = 1: fica %d\n", roda(100000, 1, 0));
    return 0;
}

O mesmo `batata_tad.c` compilado com cada implementação. As duas saídas devem ser iguais — e nenhuma
das duas tem `MAX`. Para $k = 1$ há fórmula: escreva $n = 2^m + l$, com $0 \le l < 2^m$; sobra $2l + 1$.
Para $100\,000 = 2^{16} + 34\,464$, sobra $68\,929$.

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra batata_tad.c fila_vetor.c -o batata_vetor && ./batata_vetor
!echo ---
!gcc -g -fsanitize=address -Wall -Wextra batata_tad.c fila_lista.c -o batata_lista && ./batata_lista

### Exercício 4 — a fila numa lista

Implemente `enfileira` (insere depois de `fim`) e `desenfileira` (remove o primeiro; encerra com mensagem
se a fila estiver vazia). Não esqueça o caso da fila que esvazia. O teste usa a sua fila na roda da
batata, roda com `-fsanitize=address` e, no Linux, também acusa memória que não foi liberada.

In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

typedef struct {
    No *ini;   /* a frente: primeiro no, ou NULL */
    No *fim;   /* o ultimo no, ou NULL */
} Fila;

/* Exercicio 4: enfileira insere x depois do ultimo no; desenfileira remove
   o primeiro no e devolve o valor (com a fila vazia, imprime uma mensagem em
   stderr e encerra). Quando a fila esvazia, ini e fim viram NULL.
   TODO: implemente enfileira e desenfileira. */
void enfileira(Fila *f, int x)
{
    /* TODO: implemente aqui */
    (void) f;
    (void) x;
}

int desenfileira(Fila *f)
{
    /* TODO: implemente aqui */
    (void) f;
    return 0;
}

int vazia(const Fila *f)
{
    return f->ini == NULL;
}

int roda(int n, int k)
{
    Fila f = {NULL, NULL};
    for (int i = 1; i <= n; i++)
        enfileira(&f, i);
    while (f.ini != f.fim) {          /* mais de um na roda */
        for (int j = 0; j < k; j++)
            enfileira(&f, desenfileira(&f));
        desenfileira(&f);
    }
    return desenfileira(&f);
}

int main(void)
{
    Fila f = {NULL, NULL};
    enfileira(&f, 1);
    enfileira(&f, 2);
    printf("sai %d\n", desenfileira(&f));
    printf("sai %d\n", desenfileira(&f));
    printf("vazia? %s\n", vazia(&f) ? "sim" : "nao");
    enfileira(&f, 3);
    printf("depois de enfileirar 3, vazia? %s\n", vazia(&f) ? "sim" : "nao");
    printf("sai %d\n", desenfileira(&f));
    printf("n = 41, k = 2: fica %d\n", roda(41, 2));
    printf("n = 7, k = 1: fica %d\n", roda(7, 1));
    printf("n = 1, k = 5: fica %d\n", roda(1, 5));
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex4.c -o ex4
testa('./ex4', '''sai 1
sai 2
vazia? sim
depois de enfileirar 3, vazia? nao
sai 3
n = 41, k = 2: fica 31
n = 7, k = 1: fica 7
n = 1, k = 5: fica 1''')

## 6. Quanto custa

`custo.c` roda a batata com $n$ crianças e $k = 1$ em três filas: o vetor que **desloca** todos a cada
saída, o vetor **circular** que dobra e a **lista**. As três devem dar o mesmo sobrevivente. Antes de
executar: se $n$ dobrar, o que acontece com o tempo de cada uma?

In [ ]:
%%writefile custo.c
/* custo.c --- a roda com n criancas e k = 1 em tres filas:
   vetor que desloca tudo a cada saida, vetor circular que dobra e lista.
   uso: ./custo n */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static double agora(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

/* --- vetor que desloca: a frente e sempre v[0] --- */
typedef struct {
    int *v;
    int n;
} FilaD;

static void enfileira_d(FilaD *f, int x)
{
    f->v[f->n++] = x;
}

static int desenfileira_d(FilaD *f)
{
    int x = f->v[0];
    for (int i = 1; i < f->n; i++)     /* todos andam uma casa */
        f->v[i - 1] = f->v[i];
    f->n--;
    return x;
}

/* --- vetor circular que dobra --- */
typedef struct {
    int *v;
    int ini, n, cap;
} FilaC;

static void enfileira_c(FilaC *f, int x)
{
    if (f->n == f->cap) {
        int *novo = aloca(NULL, 2 * f->cap * sizeof *novo);
        for (int i = 0; i < f->n; i++)
            novo[i] = f->v[(f->ini + i) % f->cap];
        free(f->v);
        f->v = novo;
        f->ini = 0;
        f->cap *= 2;
    }
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

static int desenfileira_c(FilaC *f)
{
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

/* --- lista: sai do primeiro no, entra depois do ultimo --- */
typedef struct no {
    int valor;
    struct no *prox;
} No;

typedef struct {
    No *ini, *fim;
    int n;
} FilaL;

static void enfileira_l(FilaL *f, int x)
{
    No *novo = aloca(NULL, sizeof *novo);
    novo->valor = x;
    novo->prox = NULL;
    if (f->fim == NULL)
        f->ini = novo;
    else
        f->fim->prox = novo;
    f->fim = novo;
    f->n++;
}

static int desenfileira_l(FilaL *f)
{
    No *velho = f->ini;
    int x = velho->valor;
    f->ini = velho->prox;
    if (f->ini == NULL)
        f->fim = NULL;
    free(velho);
    f->n--;
    return x;
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 20000;
    int fd, fc, fl;

    double t0 = agora();
    FilaD d = {aloca(NULL, n * sizeof(int)), 0};
    for (int i = 1; i <= n; i++) enfileira_d(&d, i);
    while (d.n > 1) {
        enfileira_d(&d, desenfileira_d(&d));
        desenfileira_d(&d);
    }
    fd = desenfileira_d(&d);
    free(d.v);
    double t1 = agora();

    FilaC c = {aloca(NULL, 4 * sizeof(int)), 0, 0, 4};
    for (int i = 1; i <= n; i++) enfileira_c(&c, i);
    while (c.n > 1) {
        enfileira_c(&c, desenfileira_c(&c));
        desenfileira_c(&c);
    }
    fc = desenfileira_c(&c);
    free(c.v);
    double t2 = agora();

    FilaL l = {NULL, NULL, 0};
    for (int i = 1; i <= n; i++) enfileira_l(&l, i);
    while (l.n > 1) {
        enfileira_l(&l, desenfileira_l(&l));
        desenfileira_l(&l);
    }
    fl = desenfileira_l(&l);
    double t3 = agora();

    printf("n = %d: desloca %.4f s  circular %.4f s  lista %.4f s  (fica %d %d %d)\n",
           n, t1 - t0, t2 - t1, t3 - t2, fd, fc, fl);
    return 0;
}

In [ ]:
# Visualização: tempo da roda com k = 1 nas três filas
import matplotlib.pyplot as plt

!gcc -O2 -Wall -Wextra custo.c -o custo

ns, td, tc, tl = [], [], [], []
for n in [10000, 20000, 40000, 60000, 80000]:
    s = subprocess.run(['./custo', str(n)], capture_output=True, text=True).stdout
    print(s.strip())
    p = s.split()   # n = N: desloca TD s  circular TC s  lista TL s  (fica ...)
    ns.append(n)
    td.append(float(p[4])); tc.append(float(p[7])); tl.append(float(p[10]))

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(ns, td, 'o-', label='desloca')
a.plot(ns, tc, 's-', label='circular que dobra')
a.plot(ns, tl, '^-', label='lista')
a.set_xlabel('n (crianças)'); a.set_ylabel('tempo (s)'); a.set_title('as três filas')
a.legend()
b.plot(ns, tc, 's-', color='C1', label='circular que dobra')
b.plot(ns, tl, '^-', color='C2', label='lista')
b.set_xlabel('n (crianças)'); b.set_ylabel('tempo (s)'); b.set_title('sem a que desloca')
b.legend()
plt.tight_layout()
plt.show()

Deslocando, cada saída copia a fila inteira, e são $n$ saídas: dobrar $n$ **quadruplica** o tempo — a
curva é uma parábola. Na circular e na lista, toda operação tem custo fixo: dobrar $n$ dobra o tempo. A
lista ainda é algumas vezes mais lenta que a circular — um `malloc` e um `free` por passada da batata, e
os nós espalhados pela memória.

| | vetor que desloca | circular que dobra | lista com `fim` |
|---|---|---|---|
| `enfileira` | constante | constante (amortizado) | constante + `malloc` |
| `desenfileira` | **tamanho da fila** | constante | constante + `free` |
| pode encher? | sim, em `MAX` | só sem memória | só sem memória |
| caso delicado | — | crescer na ordem da fila | `fim` ao esvaziar |

### Exercício 5 — fila com duas pilhas

Dá para montar uma fila com duas pilhas: `entra` recebe quem é enfileirado; `sai` entrega quem é
desenfileirado. Quando `sai` está vazia e alguém precisa sair, **todos** os de `entra` são passados para
`sai` — o que inverte a ordem e põe o mais antigo no topo. Cada elemento é passado uma única vez, então o
custo amortizado por operação é constante.

Implemente `enfileira` e `desenfileira` usando só `empilha`, `desempilha` e `pvazia`. O teste roda a
batata com $n = 200\,000$: uma solução que passa os elementos de volta a cada operação não termina a
tempo.

In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <stdlib.h>

/* --- uma pilha de int num vetor que dobra (a da aula passada) --- */
typedef struct {
    int *v;
    int n, cap;
} Pilha;

void empilha(Pilha *p, int x)
{
    if (p->n == p->cap) {
        p->cap = (p->cap == 0) ? 4 : 2 * p->cap;
        p->v = realloc(p->v, p->cap * sizeof *p->v);
        if (p->v == NULL) { perror("realloc"); exit(1); }
    }
    p->v[p->n++] = x;
}

int desempilha(Pilha *p)
{
    if (p->n == 0) {
        fprintf(stderr, "desempilha: pilha vazia\n");
        exit(1);
    }
    return p->v[--p->n];
}

int pvazia(const Pilha *p)
{
    return p->n == 0;
}

/* --- a fila: duas pilhas --- */
typedef struct {
    Pilha entra, sai;
} Fila;

/* Exercicio 5: enfileira empilha em entra; desenfileira desempilha de sai,
   mas, se sai estiver vazia, antes passa todos de entra para sai.
   TODO: implemente enfileira e desenfileira. */
void enfileira(Fila *f, int x)
{
    /* TODO: implemente aqui */
    (void) f;
    (void) x;
}

int desenfileira(Fila *f)
{
    /* TODO: implemente aqui */
    (void) f;
    return 0;
}

int tamanho(const Fila *f)
{
    return f->entra.n + f->sai.n;
}

int roda(int n, int k)
{
    Fila f = {{NULL, 0, 0}, {NULL, 0, 0}};
    for (int i = 1; i <= n; i++)
        enfileira(&f, i);
    while (tamanho(&f) > 1) {
        for (int j = 0; j < k; j++)
            enfileira(&f, desenfileira(&f));
        desenfileira(&f);
    }
    int fica = desenfileira(&f);
    free(f.entra.v);
    free(f.sai.v);
    return fica;
}

int main(void)
{
    Fila f = {{NULL, 0, 0}, {NULL, 0, 0}};
    for (int i = 1; i <= 3; i++)
        enfileira(&f, i);
    printf("sai %d\n", desenfileira(&f));
    enfileira(&f, 4);
    enfileira(&f, 5);
    printf("sai:");
    while (tamanho(&f) > 0)
        printf(" %d", desenfileira(&f));
    printf("\n");
    free(f.entra.v);
    free(f.sai.v);
    printf("n = 41, k = 2: fica %d\n", roda(41, 2));
    printf("n = 200000, k = 1: fica %d\n", roda(200000, 1));
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra ex5.c -o ex5
testa('./ex5', '''sai 1
sai: 2 3 4 5
n = 41, k = 2: fica 31
n = 200000, k = 1: fica 137857''', tempo=10)

## Tarefas: altere o programa da aula

Volte à célula de `batata.c`, faça cada alteração, reexecute e confira se a saída bate com o que você
previu.

1. **O menor `MAX`.** Qual é o menor `MAX` com que `./batata 41 2` termina? E `./batata 41 0`? Escreva a
   fórmula em função de $n$ e $k$ antes de conferir.
2. **Uma batata que acelera.** Faça a batata passar um número diferente de vezes a cada rodada: na rodada
   $r$ (a partir de 1), ela passa $r$ vezes. Preveja a ordem de saída para $n = 5$ antes de executar.
3. **Circular no limite.** Em `batata_circular.c`, troque `MAX` por 40 e rode `./batata_circular 41 2`.
   Em que momento o programa para? Por que não para antes?

## Desafio: o caminho mais curto no labirinto

Leia um labirinto da entrada padrão — `#` parede, `.` livre, `E` entrada, `S` saída; no máximo 50 linhas
de 80 colunas — e imprima `passos: <d>`, o menor número de passos de `E` até `S` andando nas quatro
direções, ou `sem caminho`.

Use uma fila de casas: enfileire `E` com distância 0; tire uma casa da frente e enfileire cada vizinha
livre ainda não visitada, com a distância da atual mais 1. Como a fila atende por ordem de chegada, todas
as casas a distância $d$ saem antes de qualquer uma a distância $d + 1$ — por isso, a primeira vez que
`S` sai da fila, a distância dela é a menor. Esse é o algoritmo de **busca em largura**.

In [ ]:
%%writefile labirinto.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define LIN 50
#define COL 82

char lab[LIN][COL];   /* o labirinto, uma cadeia por linha */
int nlin;             /* quantas linhas foram lidas */

void le_labirinto(void)
{
    nlin = 0;
    while (nlin < LIN && fgets(lab[nlin], COL, stdin) != NULL) {
        lab[nlin][strcspn(lab[nlin], "\n")] = '\0';
        nlin++;
    }
}

/* Desafio: devolve o menor numero de passos de 'E' ate 'S' (quatro
   direcoes, sem atravessar '#'), ou -1 se nao houver caminho.
   Sugestao: uma fila de casas (linha, coluna) num vetor de LIN * COL
   posicoes e uma matriz dist[LIN][COL] com -1 nas casas nao visitadas.
   Cuidado: as linhas podem ter comprimentos diferentes.
   TODO: implemente menor_caminho. */
int menor_caminho(void)
{
    /* TODO: implemente aqui */
    return -1;
}

int main(void)
{
    le_labirinto();
    int d = menor_caminho();
    if (d < 0)
        printf("sem caminho\n");
    else
        printf("passos: %d\n", d);
    return 0;
}

In [ ]:
labirintos = {
    'o dos slides': ('#########\n#E..#...#\n#.#.#.#.#\n#.#...#S#\n#########\n', 'passos: 12'),
    'vizinhos': ('####\n#ES#\n####\n', 'passos: 1'),
    'murado': ('#######\n#E.#..#\n#..#.S#\n#######\n', 'sem caminho'),
    'dois caminhos': ('##########\n#E.......#\n#.######.#\n#.#......#\n#....#####\n####S#####\n',
                      'passos: 7'),
    'linhas curtas': ('#####\n#E..#\n#.#\n#.....S\n', 'passos: 7'),
}
!gcc -g -fsanitize=address -Wall -Wextra labirinto.c -o labirinto
for nome, (entrada, esperado) in labirintos.items():
    print(nome + ':', end=' ')
    testa('./labirinto', esperado, entrada=entrada)

Depois de passar no teste, experimente: troque a fila por uma **pilha** (tire do mesmo lado em que
põe). O programa ainda encontra um caminho? Ainda é o menor? Em `dois caminhos`, qual dos dois a pilha
encontra? E como imprimir o labirinto com o caminho marcado — de onde cada casa foi alcançada?

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão em:

- Celes, Cerqueira e Rangel (2004), *Introdução a estruturas de dados*, cap. 12 — filas em vetor
  circular e em lista; cap. 9 — tipos abstratos de dados;
- Cormen et al. (2009), *Introduction to Algorithms*, §10.1 — filas (`ENQUEUE`, `DEQUEUE`, vetor
  circular); §10.2 — listas encadeadas; §17.4 — tabelas dinâmicas; §22.2 — busca em largura;
- Knuth (1997), *The Art of Computer Programming*, vol. 1, §2.2.1 — pilhas, filas e deques; §2.2.2 —
  alocação sequencial e a fila circular;
- Feofiloff (2009), *Algoritmos em linguagem C*, cap. 5 — filas e distâncias numa rede;
- Tenenbaum, Langsam e Augenstein (1995), *Estruturas de dados usando C*, cap. 4 — filas e listas;
- Graham, Knuth e Patashnik (1994), *Concrete Mathematics*, §1.3 — o problema de Josephus;
- Tanenbaum e Bos (2015), *Modern Operating Systems*, §2.4.3 — escalonamento *round-robin*;
- Moore (1959) — o caminho mais curto num labirinto, origem da busca em largura;
- Serebryany et al. (2012) — o AddressSanitizer.